# Get statistics and visualizations of LSST alert stream and Rubin Observations

### Other useful resources include:

- [Babamul Dashboard](https://babamul.caltech.edu/dashboard)

- [All-Sky Schedule Map](https://usdf-rsp.slac.stanford.edu/obsloctap/skymap?start=now&time=24)



### Notes on running this notebook:

This notebook fetches alerts using the Babamul broker, which requires [user credentials saved in a .env file](https://github.com/boom-astro/babamul/blob/main/examples/stream-basic/.env.example). 

Babamul has periodically changed the naming schema, for example the token was previously changed from BABAMUL_API_TOKEN to BABAMUL_TOKEN, so the .env.example linked here may be incorrect.

In [ ]:
# imports

from thor.utils.fetch_alerts import (
    babamul_get_alerts, 
    load_alerts
)

# requires user to create .env file with credentials.
import dotenv
dotenv.load_dotenv()

In [ ]:
import importlib
import thor.utils.rubin_stats_visualizations
importlib.reload(thor.utils.rubin_stats_visualizations)

In [ ]:
# Example: fetch alerts with babamul
# By default alerts will be saved locally in ../data/lsse_alert_download/raw_files. 
# Provide flag save=False to avoid this, although the function can crash when retrieving 1 week + of alerts
# It should be expected to take 10+ minutes to fetch 1 week of alerts, depending on the number of alerts

start = "07-01-2026"
end = "07-02-2026"

alerts = babamul_get_alerts(
    survey="LSST",
    start_time=start,
    end_time=end,
    save=False
) 

# # or if alerts are already already fetched and saved locally:
# alerts = load_alerts("../../data/lsst_alert_download/basic_041226_062926.json.gz")

In [ ]:
# get high level statistics of alerts fetched
from thor.utils.rubin_stats_visualizations import summarize_night

summarize_night(alerts)

In [ ]:
# get high level statistics of DDF alerts
# WARNING - currently use a naive ra/dec match with catalogs that isn't fully accurate
from thor.utils.rubin_stats_visualizations import summarize_ddf_alerts

summarize_ddf_alerts(alerts)

In [ ]:
# plot historgram of any selected alert property from a given night
from thor.utils.rubin_stats_visualizations import plot_alert_property

plot_alert_property(alerts, "magpsf", bins=20, log_scale=True)

In [ ]:
# inspect a subset of alerts by selecting on a feature such as jd

filtered_jd = [a for a in alerts if 2461222 <= a.candidate.jd <= 2461222.5]
print(f'Showing {len(filtered_jd)}/{len(alerts)}')
plot_alert_property(filtered_jd, "magpsf", bins=20, log_scale=True)

In [ ]:
# heatmap with default 3.5° bins
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(alerts, heatmap=True, plot_ddf=True, title=f"LSST Alert Skymap for {start} to {end}")

In [ ]:
from thor.utils.rubin_stats_visualizations import fetch_rubin_weather

weather = fetch_rubin_weather(date_str="7-1-2026")